# Usando RAG local para análise de um artigo científico

- Extrai conteúdo de PDFs

- Prepara informações e exporta como parquet

## Carrega bibliotecas

In [ ]:
import os
import re

from pyspark.sql import SparkSession
from pyspark.sql.functions import udf, split, size, col, length
from pyspark.sql.types import StringType

import fitz

## Define constantes

In [ ]:
INPUT_DOCS_PATH = r'../data/raw/'

## Inicia sessão Spark

In [ ]:
spark = SparkSession.builder.getOrCreate()

Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/04/06 15:08:37 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


## Extrai conteúdo de PDFs

In [ ]:
df_pdfs = (
    spark.read
    .format('binaryFile')
    .load(INPUT_DOCS_PATH)
)
df_pdfs.show()

+--------------------+--------------------+------+--------------------+
|                path|    modificationTime|length|             content|
+--------------------+--------------------+------+--------------------+
|file:/media/msc/9...|2026-04-06 13:44:...|260227|[25 50 44 46 2D 3...|
|file:/media/msc/9...|2026-04-06 13:44:...|210504|[25 50 44 46 2D 3...|
|file:/media/msc/9...|2026-04-06 13:44:...|209578|[25 50 44 46 2D 3...|
|file:/media/msc/9...|2026-04-06 13:44:...|208445|[25 50 44 46 2D 3...|
|file:/media/msc/9...|2026-04-06 13:44:...|207050|[25 50 44 46 2D 3...|
|file:/media/msc/9...|2026-04-06 13:44:...|198422|[25 50 44 46 2D 3...|
|file:/media/msc/9...|2026-04-06 13:44:...|198378|[25 50 44 46 2D 3...|
|file:/media/msc/9...|2026-04-06 13:44:...|195992|[25 50 44 46 2D 3...|
|file:/media/msc/9...|2026-04-06 13:44:...|195812|[25 50 44 46 2D 3...|
|file:/media/msc/9...|2026-04-06 13:44:...|193133|[25 50 44 46 2D 3...|
|file:/media/msc/9...|2026-04-06 13:44:...|187024|[25 50 44 46 2

In [ ]:
def clean_text(text):
    if not text:
        return text
    
    text = re.sub(r'([.,])([A-Za-z])', r'\1 \2', text)
    text = re.sub(r'\s+', ' ', text)
    
    return text.strip()


def extract_pdf_text(content):
    try:
        doc = fitz.open(stream=content, filetype='pdf')
        text = []
        for page in doc:
            text.append(page.get_text('text'))
        return clean_text('\n'.join(text))
    except Exception:
        return None

In [6]:
extract_udf = udf(extract_pdf_text, StringType())

df_parsed_docs = df_pdfs.withColumn(
    'parsed_doc',
    extract_udf('content')
)

df_parsed_docs = df_parsed_docs.withColumn(
    'file_name',
    split(col('path'), '/')[size(split(col('path'), '/')) - 1]
)

df_parsed_docs = df_parsed_docs.drop('path', 'content')

df_parsed_docs = df_parsed_docs.withColumn(
    'n_chars_parsed_doc',
    length(col('parsed_doc'))
).withColumn(
    'n_words_parsed_doc',
    size(split(col('parsed_doc'), r'\s+'))
)

In [7]:
df_parsed_docs.select('file_name', 'parsed_doc').show(truncate=False)

+-----------+---------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------

In [8]:
df_parsed_docs.toPandas()

,modificationTime,length,parsed_doc,file_name,n_chars_parsed_doc,n_words_parsed_doc
0,2026-04-06 13:44:18.396,260227,prior b′′ where Ui(S) = Ui(D) (in which case s...,pg_0032.pdf,3269,649
1,2026-04-06 13:44:18.365,210504,"for example, evidence from Facebook in Altay e...",pg_0010.pdf,3132,561
2,2026-04-06 13:44:18.370,209578,misinformation is less likely (Altay et al. (2...,pg_0013.pdf,3477,557
3,2026-04-06 13:44:18.394,208445,A Proofs A.1 Auxiliary Lemmas We deﬁne a (mixe...,pg_0031.pdf,2637,602
4,2026-04-06 13:44:18.398,207050,"(iii) U(b∗, b∗∗) i (S) = 0 for some prior b′ f...",pg_0033.pdf,2673,657
5,2026-04-06 13:44:18.361,198422,"(iii) If ν = T (the article is truthful), then...",pg_0008.pdf,2036,392
6,2026-04-06 13:44:18.399,198378,"because ˜u > ˜c. Thus, as r →0, ignoring is a ...",pg_0034.pdf,2854,528
7,2026-04-06 13:44:18.372,195992,"An important advantage of island networks, in ...",pg_0014.pdf,3493,590
8,2026-04-06 13:44:18.382,195812,now brieﬂy discuss four distinct types of regu...,pg_0020.pdf,3435,531
9,2026-04-06 13:44:18.385,193133,"PN i=1(1 −ρ)i h Qi j=1 ζj i . In general, this...",pg_0022.pdf,3403,529


## Exporta como parquet

In [9]:
df_parsed_docs.write.mode('overwrite').parquet('../data/interim/article_parsed_infos')

### Teste

In [10]:
df_test = spark.read.parquet('../data/interim/article_parsed_infos')
df_test.show()

+--------------------+------+--------------------+-----------+------------------+------------------+
|    modificationTime|length|          parsed_doc|  file_name|n_chars_parsed_doc|n_words_parsed_doc|
+--------------------+------+--------------------+-----------+------------------+------------------+
|2026-04-06 13:44:...| 47298|a strategic princ...|pg_0006.pdf|              4151|               608|
|2026-04-06 13:44:...| 34563|platform chooses ...|pg_0026.pdf|              3712|               540|
|2026-04-06 13:44:...| 34171|component. Given ...|pg_0004.pdf|              3823|               568|
|2026-04-06 13:44:...|169334|Divisiveness of C...|pg_0016.pdf|              3414|               562|
|2026-04-06 13:44:...|166820|Proof of Proposit...|pg_0042.pdf|              3708|               611|
|2026-04-06 13:44:...|159512|more likely to be...|pg_0023.pdf|              3556|               539|
|2026-04-06 13:44:...|170525|Reputability and ...|pg_0012.pdf|              3641|          

## Interrompe sessão Spark

In [11]:
spark.stop()